In [4]:

import json
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 90)
print("DATA CLEANING AND VALIDATION")
print("=" * 90)

# Project paths

PROJECT_ROOT = Path(
    r"C:\Users\hp\Analysis in Global Supply Chain Operations"
)

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "APL_Logistics.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_FILE = PROCESSED_DIR / "cleaned_logistics_data.csv"
MISSING_REPORT_FILE = REPORT_DIR / "missing_value_report.csv"
DUPLICATE_REPORT_FILE = REPORT_DIR / "duplicate_validation_report.csv"
INVALID_REPORT_FILE = REPORT_DIR / "invalid_records_report.csv"
QUALITY_REPORT_FILE = REPORT_DIR / "data_quality_validation_report.csv"
SUMMARY_FILE = REPORT_DIR / "cleaning_summary.json"
OUTLIER_REPORT_FILE = REPORT_DIR / "financial_outlier_report.csv"
IDENTIFIER_REPORT_FILE = REPORT_DIR / "identifier_validation_report.csv"
NUMERIC_REPORT_FILE = REPORT_DIR / "numeric_validation_report.csv"
CATEGORY_REPORT_FILE = REPORT_DIR / "categorical_validation_report.csv"
GEOGRAPHIC_REPORT_FILE = REPORT_DIR / "geographic_validation_report.csv"
DURATION_REPORT_FILE = REPORT_DIR / "shipping_duration_report.csv"
INVESTIGATION_REPORT_FILE = REPORT_DIR / "missing_customer_investigation_report.csv"

# Expected source columns

REQUIRED_COLUMNS = [
    "Type",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Benefit per order",
    "Sales per customer",
    "Delivery Status",
    "Late_delivery_risk",
    "Category Id",
    "Category Name",
    "Customer City",
    "Customer Country",
    "Customer Fname",
    "Customer Id",
    "Customer Lname",
    "Customer Segment",
    "Customer State",
    "Customer Street",
    "Customer Zipcode",
    "Department Id",
    "Department Name",
    "Latitude",
    "Longitude",
    "Market",
    "Order City",
    "Order Country",
    "Order Customer Id",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Profit Ratio",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Order Region",
    "Order State",
    "Order Status",
    "Product Name",
    "Product Price",
    "Shipping Mode"
]

NUMERIC_COLUMNS = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Benefit per order",
    "Sales per customer",
    "Late_delivery_risk",
    "Category Id",
    "Customer Id",
    "Customer Zipcode",
    "Department Id",
    "Latitude",
    "Longitude",
    "Order Customer Id",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Profit Ratio",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Product Price"
]

CATEGORICAL_COLUMNS = [
    "Delivery Status",
    "Shipping Mode",
    "Customer Segment",
    "Market",
    "Order Region",
    "Order Country",
    "Order State",
    "Category Name",
    "Department Name"
]

PERSONAL_COLUMNS = [
    "Customer Fname",
    "Customer Lname",
    "Customer Id",
    "Order Customer Id",
    "Customer Street",
    "Customer Zipcode",
    "Customer City",
    "Customer State",
    "Customer Country"
]

ESSENTIAL_DELIVERY_COLUMNS = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk"
]

# Load source dataset

print("\nSOURCE DATASET INFORMATION")
print("-" * 90)

if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Source dataset not found: {RAW_FILE}"
    )

df_original = pd.read_excel(RAW_FILE)

original_count = len(df_original)
original_columns = len(df_original.columns)

missing_required_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in df_original.columns
]

print(f"Dataset path: {RAW_FILE}")
print(f"Original records: {original_count:,}")
print(f"Original columns: {original_columns}")
print(f"Required columns found: {len(REQUIRED_COLUMNS) - len(missing_required_columns)}")
print(f"Missing required columns: {missing_required_columns}")

if missing_required_columns:
    raise ValueError(
        "Required source columns are missing. "
        "Cleaning cannot continue safely."
    )

df = df_original.copy()
df.insert(0, "_source_row_id", np.arange(len(df)))

# Missing value analysis

print("\nMISSING VALUE ANALYSIS")
print("-" * 90)

missing_report = pd.DataFrame({
    "Column": df_original.columns,
    "Missing Count": [
        int(df_original[column].isna().sum())
        for column in df_original.columns
    ],
    "Missing Percentage": [
        round(
            df_original[column].isna().mean() * 100,
            4
        )
        for column in df_original.columns
    ],
    "Non-Missing Count": [
        int(df_original[column].notna().sum())
        for column in df_original.columns
    ],
    "Data Type": [
        str(df_original[column].dtype)
        for column in df_original.columns
    ]
}).sort_values(
    "Missing Count",
    ascending=False
).reset_index(drop=True)

display(missing_report)

missing_report.to_csv(
    MISSING_REPORT_FILE,
    index=False
)

print(f"Missing-value report saved: {MISSING_REPORT_FILE}")

# Missing customer information investigation

print("\nMISSING CUSTOMER INFORMATION INVESTIGATION")
print("-" * 90)

lname_missing_count = int(
    df_original["Customer Lname"].isna().sum()
)

zipcode_missing_count = int(
    df_original["Customer Zipcode"].isna().sum()
)

investigation_parts = []

for column in ["Customer Lname", "Customer Zipcode"]:
    missing_ids = df.loc[
        df[column].isna(),
        ["_source_row_id"]
    ].copy()

    if not missing_ids.empty:
        missing_ids["Missing Field"] = column
        investigation_parts.append(missing_ids)

if investigation_parts:
    investigation_report = pd.concat(
        investigation_parts,
        ignore_index=True
    )
else:
    investigation_report = pd.DataFrame(
        columns=["_source_row_id", "Missing Field"]
    )

investigation_report.to_csv(
    INVESTIGATION_REPORT_FILE,
    index=False
)

print(f"Missing Customer Lname records: {lname_missing_count}")
print(f"Missing Customer Zipcode records: {zipcode_missing_count}")
print("Customer personal information has not been reproduced in the investigation report.")
print(f"Privacy-safe investigation report saved: {INVESTIGATION_REPORT_FILE}")

print("\nMISSING VALUE HANDLING")
print("-" * 90)

print("Missing customer names and ZIP codes are retained in the source audit.")
print("No artificial customer information has been generated.")
print("Missing analytical fields are evaluated separately.")
print("No records are removed solely because customer descriptive information is missing.")

# Duplicate record validation

print("\nDUPLICATE RECORD VALIDATION")
print("-" * 90)

duplicate_mask = df_original.duplicated(
    subset=REQUIRED_COLUMNS,
    keep="first"
)

duplicate_count = int(duplicate_mask.sum())

duplicate_ids = df.loc[
    duplicate_mask,
    ["_source_row_id"]
].copy()

duplicate_ids["Validation Result"] = "Exact duplicate record"

duplicate_ids.to_csv(
    DUPLICATE_REPORT_FILE,
    index=False
)

records_after_duplicates = original_count - duplicate_count

df = df.loc[
    ~duplicate_mask
].copy().reset_index(drop=True)

print(f"Original record count: {original_count:,}")
print(f"Exact duplicate records identified: {duplicate_count:,}")
print(f"Records retained after duplicate removal: {len(df):,}")
print(f"Duplicate validation report saved: {DUPLICATE_REPORT_FILE}")

# Customer and order identifier validation

print("\nCUSTOMER AND ORDER IDENTIFIER VALIDATION")
print("-" * 90)

identifier_rows = []

for column in ["Customer Id", "Order Customer Id"]:
    identifier_rows.append({
        "Identifier": column,
        "Missing Values": int(df[column].isna().sum()),
        "Unique Values": int(df[column].nunique(dropna=True)),
        "Repeated Records": int(df[column].duplicated().sum()),
        "Unique Identifier Check": (
            "UNIQUE"
            if df[column].is_unique
            else "REPEATED CUSTOMER IDENTIFIERS"
        )
    })

identifier_mismatch_count = int(
    (
        df["Customer Id"].notna()
        & df["Order Customer Id"].notna()
        & (df["Customer Id"] != df["Order Customer Id"])
    ).sum()
)

identifier_rows.append({
    "Identifier": "Customer Id vs Order Customer Id",
    "Missing Values": int(
        (
            df["Customer Id"].isna()
            | df["Order Customer Id"].isna()
        ).sum()
    ),
    "Unique Values": np.nan,
    "Repeated Records": np.nan,
    "Unique Identifier Check": (
        f"{identifier_mismatch_count} mismatches"
    )
})

identifier_report = pd.DataFrame(identifier_rows)

display(identifier_report)

identifier_report.to_csv(
    IDENTIFIER_REPORT_FILE,
    index=False
)

print(
    "Important: The source dataset does not contain a dedicated "
    "Order ID column. Customer identifiers are not treated as unique "
    "order identifiers."
)

# Numeric data type validation

print("\nNUMERIC DATA TYPE VALIDATION")
print("-" * 90)

numeric_validation_rows = []
numeric_conversion_masks = {}

for column in NUMERIC_COLUMNS:

    original_dtype = str(df[column].dtype)

    original_nonmissing = df[column].notna()

    converted = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    conversion_failure_mask = (
        original_nonmissing & converted.isna()
    )

    numeric_conversion_masks[column] = conversion_failure_mask

    df[column] = converted

    numeric_validation_rows.append({
        "Column": column,
        "Original Data Type": original_dtype,
        "Final Data Type": str(df[column].dtype),
        "Conversion Failures": int(
            conversion_failure_mask.sum()
        ),
        "Missing Values After Conversion": int(
            df[column].isna().sum()
        )
    })

numeric_report = pd.DataFrame(
    numeric_validation_rows
)

display(numeric_report)

numeric_report.to_csv(
    NUMERIC_REPORT_FILE,
    index=False
)

# Invalid numeric and non-finite value validation

print("\nINVALID NUMERIC VALUE VALIDATION")
print("-" * 90)

invalid_numeric_rows = []

for column in NUMERIC_COLUMNS:

    conversion_failures = int(
        numeric_conversion_masks[column].sum()
    )

    values = df[column].to_numpy(
        dtype=float,
        na_value=np.nan
    )

    non_finite_count = int(
        np.isinf(values).sum()
    )

    invalid_numeric_rows.append({
        "Column": column,
        "Conversion Failures": conversion_failures,
        "Infinite Values": non_finite_count
    })

    print(
        f"{column}: "
        f"{conversion_failures} conversion failures, "
        f"{non_finite_count} infinite values"
    )

    if non_finite_count:
        df.loc[
            np.isinf(values),
            column
        ] = np.nan

invalid_numeric_report = pd.DataFrame(
    invalid_numeric_rows
)

invalid_numeric_report.to_csv(
    REPORT_DIR / "invalid_numeric_values_report.csv",
    index=False
)

# Shipping duration validation

print("\nSHIPPING DURATION VALIDATION")
print("-" * 90)

actual_column = "Days for shipping (real)"
scheduled_column = "Days for shipment (scheduled)"

actual_negative_mask = df[actual_column] < 0
scheduled_negative_mask = df[scheduled_column] < 0

actual_missing_mask = df[actual_column].isna()
scheduled_missing_mask = df[scheduled_column].isna()

zero_scheduled_mask = (
    df[scheduled_column].eq(0)
)

same_day_zero_mask = (
    zero_scheduled_mask
    & df["Shipping Mode"].astype(str).str.strip().str.casefold().eq("same day")
)

other_zero_mask = (
    zero_scheduled_mask
    & ~same_day_zero_mask
)

def calculate_iqr_outliers(series):
    valid = series.dropna()

    if len(valid) == 0:
        return pd.Series(
            False,
            index=series.index
        ), np.nan, np.nan

    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (
        (series < lower)
        | (series > upper)
    ).fillna(False)

    return mask, float(lower), float(upper)

actual_extreme_mask, actual_lower, actual_upper = (
    calculate_iqr_outliers(df[actual_column])
)

scheduled_extreme_mask, scheduled_lower, scheduled_upper = (
    calculate_iqr_outliers(df[scheduled_column])
)

duration_report = pd.DataFrame([
    {
        "Validation": "Missing actual shipping duration",
        "Count": int(actual_missing_mask.sum())
    },
    {
        "Validation": "Missing scheduled shipping duration",
        "Count": int(scheduled_missing_mask.sum())
    },
    {
        "Validation": "Negative actual shipping duration",
        "Count": int(actual_negative_mask.sum())
    },
    {
        "Validation": "Negative scheduled shipping duration",
        "Count": int(scheduled_negative_mask.sum())
    },
    {
        "Validation": "Zero scheduled shipping duration",
        "Count": int(zero_scheduled_mask.sum())
    },
    {
        "Validation": "Zero scheduled duration with Same Day mode",
        "Count": int(same_day_zero_mask.sum())
    },
    {
        "Validation": "Zero scheduled duration with other modes",
        "Count": int(other_zero_mask.sum())
    },
    {
        "Validation": "Extreme actual shipping durations by IQR",
        "Count": int(actual_extreme_mask.sum())
    },
    {
        "Validation": "Extreme scheduled shipping durations by IQR",
        "Count": int(scheduled_extreme_mask.sum())
    }
])

display(duration_report)

duration_report.to_csv(
    DURATION_REPORT_FILE,
    index=False
)

print("\nZERO SCHEDULED DURATION INVESTIGATION")
print("-" * 90)

print(
    f"Zero scheduled duration records: "
    f"{int(zero_scheduled_mask.sum()):,}"
)

zero_duration_investigation = df.loc[
    zero_scheduled_mask,
    [
        "_source_row_id",
        actual_column,
        scheduled_column,
        "Shipping Mode",
        "Delivery Status",
        "Late_delivery_risk"
    ]
].copy()

display(
    zero_duration_investigation.head(20)
)

zero_duration_investigation.to_csv(
    REPORT_DIR / "zero_scheduled_duration_investigation.csv",
    index=False
)

print(
    "Zero scheduled durations are retained and investigated. "
    "They are not automatically classified as invalid."
)

# Categorical data standardization

print("\nCATEGORICAL DATA STANDARDIZATION")
print("-" * 90)

def normalize_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value)
    value = re.sub(r"\s+", " ", value).strip()

    if value == "":
        return np.nan

    return value

def canonicalize_column(series):
    cleaned = series.map(normalize_text)

    nonmissing = cleaned.dropna()

    canonical_lookup = {}

    for value in nonmissing.unique():
        key = value.casefold()

        if key not in canonical_lookup:
            canonical_lookup[key] = value

    return cleaned.map(
        lambda value: (
            canonical_lookup.get(value.casefold(), value)
            if isinstance(value, str)
            else np.nan
        )
    )

for column in CATEGORICAL_COLUMNS:
    df[column] = canonicalize_column(
        df[column]
    )

# Normalize market labels consistently

market_map = {
    "latam": "LATAM",
    "europe": "Europe",
    "pacific asia": "Pacific Asia",
    "usca": "USCA",
    "africa": "Africa"
}

df["Market"] = df["Market"].map(
    lambda value: (
        market_map.get(value.casefold(), value)
        if isinstance(value, str)
        else np.nan
    )
)

categorical_validation_rows = []

for column in CATEGORICAL_COLUMNS:
    categorical_validation_rows.append({
        "Column": column,
        "Unique Values After Standardization": int(
            df[column].nunique(dropna=True)
        ),
        "Missing Values": int(
            df[column].isna().sum()
        ),
        "Blank Values Converted to Missing": int(
            df[column].isna().sum()
        )
    })

categorical_report = pd.DataFrame(
    categorical_validation_rows
)

display(categorical_report)

categorical_report.to_csv(
    CATEGORY_REPORT_FILE,
    index=False
)

for column in CATEGORICAL_COLUMNS:
    print(f"\n{column}")
    print(
        df[column]
        .value_counts(dropna=False)
        .head(20)
    )

# Geographic coordinate validation

print("\nGEOGRAPHIC COORDINATE VALIDATION")
print("-" * 90)

latitude_missing = df["Latitude"].isna()
longitude_missing = df["Longitude"].isna()

latitude_invalid = (
    ~df["Latitude"].between(-90, 90)
).fillna(False)

longitude_invalid = (
    ~df["Longitude"].between(-180, 180)
).fillna(False)

invalid_geographic_mask = (
    latitude_missing
    | longitude_missing
    | latitude_invalid
    | longitude_invalid
)

geographic_report = pd.DataFrame([
    {
        "Validation": "Missing latitude",
        "Count": int(latitude_missing.sum())
    },
    {
        "Validation": "Latitude outside valid range",
        "Count": int(latitude_invalid.sum())
    },
    {
        "Validation": "Missing longitude",
        "Count": int(longitude_missing.sum())
    },
    {
        "Validation": "Longitude outside valid range",
        "Count": int(longitude_invalid.sum())
    },
    {
        "Validation": "Invalid geographic coordinate records",
        "Count": int(invalid_geographic_mask.sum())
    }
])

display(geographic_report)

geographic_report.to_csv(
    GEOGRAPHIC_REPORT_FILE,
    index=False
)

print(
    "Geographic note: Latitude and Longitude represent customer "
    "coordinates in this dataset. They are not confirmed order "
    "destination coordinates."
)

# Late delivery risk validation

print("\nLATE DELIVERY RISK VALIDATION")
print("-" * 90)

risk_numeric = pd.to_numeric(
    df["Late_delivery_risk"],
    errors="coerce"
)

invalid_risk_mask = (
    risk_numeric.isna()
    | ~risk_numeric.isin([0, 1])
)

print(
    f"Valid risk values: "
    f"{sorted(risk_numeric.dropna().unique().tolist())}"
)

print(
    f"Invalid or missing risk records: "
    f"{int(invalid_risk_mask.sum()):,}"
)

# Financial data validation

print("\nFINANCIAL DATA VALIDATION")
print("-" * 90)

financial_columns = [
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Benefit per order",
    "Order Item Quantity"
]

financial_rows = []

for column in financial_columns:
    series = df[column]

    financial_rows.append({
        "Column": column,
        "Missing Values": int(series.isna().sum()),
        "Minimum": series.min(),
        "Maximum": series.max(),
        "Negative Values": int((series < 0).sum()),
        "Zero Values": int((series == 0).sum())
    })

financial_report = pd.DataFrame(
    financial_rows
)

display(financial_report)

print("\nFINANCIAL VALIDATION RESULTS")
print(
    f"Negative Sales values: "
    f"{int((df['Sales'] < 0).sum()):,}"
)

print(
    f"Zero Sales values: "
    f"{int((df['Sales'] == 0).sum()):,}"
)

print(
    f"Negative Order Item Total values: "
    f"{int((df['Order Item Total'] < 0).sum()):,}"
)

print(
    f"Zero Order Item Total values: "
    f"{int((df['Order Item Total'] == 0).sum()):,}"
)

print(
    f"Negative Order Profit Per Order values: "
    f"{int((df['Order Profit Per Order'] < 0).sum()):,}"
)

print(
    f"Negative Benefit per order values: "
    f"{int((df['Benefit per order'] < 0).sum()):,}"
)

quantity_invalid_mask = (
    df["Order Item Quantity"].isna()
    | (df["Order Item Quantity"] <= 0)
    | (
        df["Order Item Quantity"].notna()
        & (
            df["Order Item Quantity"]
            != np.floor(df["Order Item Quantity"])
        )
    )
)

print(
    f"Invalid Order Item Quantity values: "
    f"{int(quantity_invalid_mask.sum()):,}"
)

# Financial outlier analysis

print("\nFINANCIAL OUTLIER ANALYSIS")
print("-" * 90)

outlier_columns = [
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Benefit per order"
]

financial_outlier_masks = {}
outlier_rows = []

for column in outlier_columns:

    mask, lower, upper = calculate_iqr_outliers(
        df[column]
    )

    financial_outlier_masks[column] = mask

    outlier_rows.append({
        "Column": column,
        "Outlier Count": int(mask.sum()),
        "Lower IQR Bound": lower,
        "Upper IQR Bound": upper,
        "Minimum": df[column].min(),
        "Maximum": df[column].max()
    })

outlier_report = pd.DataFrame(
    outlier_rows
)

display(outlier_report)

outlier_report.to_csv(
    OUTLIER_REPORT_FILE,
    index=False
)

print(
    "Financial outliers are flagged for investigation and retained. "
    "Negative profits are not automatically removed."
)

# Validation flags and invalid record documentation

print("\nINVALID RECORD IDENTIFICATION")
print("-" * 90)

flag_columns = {}

for column in NUMERIC_COLUMNS:
    flag_columns[f"Invalid numeric: {column}"] = (
        numeric_conversion_masks[column]
    )

flag_columns["Infinite numeric values"] = pd.Series(
    False,
    index=df.index
)

for column in NUMERIC_COLUMNS:
    values = df[column].to_numpy(
        dtype=float,
        na_value=np.nan
    )
    flag_columns["Infinite numeric values"] |= pd.Series(
        np.isinf(values),
        index=df.index
    )

flag_columns["Negative actual shipping duration"] = (
    df[actual_column] < 0
).fillna(False)

flag_columns["Negative scheduled shipping duration"] = (
    df[scheduled_column] < 0
).fillna(False)

flag_columns["Invalid delivery status"] = (
    df["Delivery Status"].isna()
    | ~df["Delivery Status"].isin([
        "Advance shipping",
        "Late delivery",
        "Shipping on time",
        "Shipping canceled"
    ])
)

flag_columns["Invalid shipping mode"] = (
    df["Shipping Mode"].isna()
    | ~df["Shipping Mode"].isin([
        "First Class",
        "Second Class",
        "Standard Class",
        "Same Day"
    ])
)

flag_columns["Invalid customer segment"] = (
    df["Customer Segment"].isna()
    | ~df["Customer Segment"].isin([
        "Consumer",
        "Corporate",
        "Home Office"
    ])
)

flag_columns["Invalid market"] = (
    df["Market"].isna()
    | ~df["Market"].isin([
        "Africa",
        "Europe",
        "LATAM",
        "Pacific Asia",
        "USCA"
    ])
)

flag_columns["Invalid late delivery risk"] = (
    invalid_risk_mask
)

flag_columns["Invalid geographic coordinates"] = (
    invalid_geographic_mask
)

flag_columns["Invalid order quantity"] = (
    quantity_invalid_mask
)

flag_columns["Zero scheduled duration"] = (
    zero_scheduled_mask
)

for column, mask in financial_outlier_masks.items():
    flag_columns[f"Financial outlier: {column}"] = mask

for column in [
    "Sales",
    "Order Item Total"
]:
    flag_columns[f"Negative or zero value: {column}"] = (
        (df[column] <= 0).fillna(False)
    )

flag_frame = pd.DataFrame(
    flag_columns,
    index=df.index
).fillna(False).astype(bool)

records_with_flags_mask = flag_frame.any(axis=1)

invalid_records = df.loc[
    records_with_flags_mask,
    [
        "_source_row_id",
        "Delivery Status",
        "Shipping Mode",
        "Market",
        "Order Region",
        "Days for shipping (real)",
        "Days for shipment (scheduled)",
        "Late_delivery_risk",
        "Sales",
        "Order Item Total",
        "Order Profit Per Order",
        "Benefit per order",
        "Order Item Quantity"
    ]
].copy()

invalid_records["Validation Issues"] = [
    "; ".join(
        flag_frame.columns[
            flag_frame.loc[index].to_numpy()
        ].tolist()
    )
    for index in invalid_records.index
]

invalid_records.to_csv(
    INVALID_REPORT_FILE,
    index=False
)

print(
    f"Records with one or more validation flags: "
    f"{int(records_with_flags_mask.sum()):,}"
)

print(
    f"Privacy-safe invalid records report saved: "
    f"{INVALID_REPORT_FILE}"
)

# Critical delivery record validation

print("\nCRITICAL DELIVERY RECORD VALIDATION")
print("-" * 90)

critical_invalid_mask = (
    df[actual_column].isna()
    | df[scheduled_column].isna()
    | (df[actual_column] < 0)
    | (df[scheduled_column] < 0)
    | df["Delivery Status"].isna()
    | ~df["Delivery Status"].isin([
        "Advance shipping",
        "Late delivery",
        "Shipping on time",
        "Shipping canceled"
    ])
    | invalid_risk_mask
)

critical_invalid_count = int(
    critical_invalid_mask.sum()
)

df_clean = df.loc[
    ~critical_invalid_mask
].copy().reset_index(drop=True)

print(
    f"Critical invalid records excluded: "
    f"{critical_invalid_count:,}"
)

print(
    f"Records remaining: {len(df_clean):,}"
)

# Remove personal identifiers from analytical outputs

print("\nPERSONAL IDENTIFIER REMOVAL")
print("-" * 90)

removed_personal_columns = [
    column for column in PERSONAL_COLUMNS
    if column in df_clean.columns
]

df_clean.drop(
    columns=removed_personal_columns,
    inplace=True
)

print("Personal and customer-identifying columns excluded:")

for column in removed_personal_columns:
    print(column)

# Generate cleaned analytical dataset

print("\nCLEANED DATASET GENERATION")
print("-" * 90)

df_clean.to_csv(
    CLEANED_FILE,
    index=False
)

print(f"Cleaned dataset saved: {CLEANED_FILE}")
print(f"Cleaned dataset records: {len(df_clean):,}")
print(f"Cleaned dataset columns: {len(df_clean.columns)}")

# Final data quality validation

print("\nFINAL DATA QUALITY VALIDATION")
print("-" * 90)

quality_checks = []

def add_quality_check(name, condition, details):
    quality_checks.append({
        "Validation": name,
        "Status": "PASS" if bool(condition) else "FAIL",
        "Details": str(details)
    })

add_quality_check(
    "Required source columns available",
    len(missing_required_columns) == 0,
    f"Missing columns: {missing_required_columns}"
)

add_quality_check(
    "No exact duplicate records in cleaned data",
    not df_clean.duplicated().any(),
    f"Duplicate count: {int(df_clean.duplicated().sum())}"
)

add_quality_check(
    "Actual shipping duration is numeric",
    pd.api.types.is_numeric_dtype(df_clean[actual_column]),
    str(df_clean[actual_column].dtype)
)

add_quality_check(
    "Scheduled shipping duration is numeric",
    pd.api.types.is_numeric_dtype(df_clean[scheduled_column]),
    str(df_clean[scheduled_column].dtype)
)

add_quality_check(
    "No negative actual shipping durations",
    not (df_clean[actual_column] < 0).any(),
    f"Minimum: {df_clean[actual_column].min()}"
)

add_quality_check(
    "No negative scheduled shipping durations",
    not (df_clean[scheduled_column] < 0).any(),
    f"Minimum: {df_clean[scheduled_column].min()}"
)

add_quality_check(
    "Late delivery risk contains only valid values",
    df_clean["Late_delivery_risk"].notna().all()
    and df_clean["Late_delivery_risk"].isin([0, 1]).all(),
    f"Values: {sorted(df_clean['Late_delivery_risk'].dropna().unique().tolist())}"
)

add_quality_check(
    "Delivery status values are standardized",
    df_clean["Delivery Status"].notna().all()
    and df_clean["Delivery Status"].isin([
        "Advance shipping",
        "Late delivery",
        "Shipping on time",
        "Shipping canceled"
    ]).all(),
    f"Values: {sorted(df_clean['Delivery Status'].dropna().unique().tolist())}"
)

add_quality_check(
    "Shipping mode values are standardized",
    df_clean["Shipping Mode"].notna().all()
    and df_clean["Shipping Mode"].isin([
        "First Class",
        "Second Class",
        "Standard Class",
        "Same Day"
    ]).all(),
    f"Values: {sorted(df_clean['Shipping Mode'].dropna().unique().tolist())}"
)

add_quality_check(
    "Customer segment values are standardized",
    df_clean["Customer Segment"].notna().all()
    and df_clean["Customer Segment"].isin([
        "Consumer",
        "Corporate",
        "Home Office"
    ]).all(),
    f"Values: {sorted(df_clean['Customer Segment'].dropna().unique().tolist())}"
)

add_quality_check(
    "Market values are standardized",
    df_clean["Market"].notna().all()
    and df_clean["Market"].isin([
        "Africa",
        "Europe",
        "LATAM",
        "Pacific Asia",
        "USCA"
    ]).all(),
    f"Values: {sorted(df_clean['Market'].dropna().unique().tolist())}"
)

add_quality_check(
    "Geographic coordinates are within valid ranges",
    (
        df_clean["Latitude"].dropna().between(-90, 90).all()
        and df_clean["Longitude"].dropna().between(-180, 180).all()
    ),
    (
        f"Invalid coordinate records retained for review: "
        f"{int(invalid_geographic_mask.loc[df_clean.index].sum()) if len(df_clean) else 0}"
    )
)

add_quality_check(
    "Order quantity is positive and integral",
    (
        df_clean["Order Item Quantity"].notna().all()
        and (df_clean["Order Item Quantity"] > 0).all()
        and (
            df_clean["Order Item Quantity"]
            == np.floor(df_clean["Order Item Quantity"])
        ).all()
    ),
    "Quantity must be a positive whole number"
)

add_quality_check(
    "No personal identifier columns in analytical dataset",
    not any(
        column in df_clean.columns
        for column in PERSONAL_COLUMNS
    ),
    f"Removed columns: {removed_personal_columns}"
)

add_quality_check(
    "No missing essential delivery fields",
    not df_clean[ESSENTIAL_DELIVERY_COLUMNS].isna().any().any(),
    "Essential delivery fields checked"
)

expected_final_count = (
    original_count
    - duplicate_count
    - critical_invalid_count
)

record_count_reconciled = (
    len(df_clean) == expected_final_count
)

add_quality_check(
    "Record counts reconciled",
    record_count_reconciled,
    (
        f"Original: {original_count}; "
        f"Duplicates removed: {duplicate_count}; "
        f"Critical exclusions: {critical_invalid_count}; "
        f"Final: {len(df_clean)}; "
        f"Expected final: {expected_final_count}"
    )
)

quality_report = pd.DataFrame(
    quality_checks
)

display(quality_report)

quality_report.to_csv(
    QUALITY_REPORT_FILE,
    index=False
)

# Cleaning summary

print("\nDATA CLEANING SUMMARY")
print("-" * 90)

records_removed_duplicates = int(duplicate_count)
records_removed_critical = int(critical_invalid_count)

records_removed_total = (
    records_removed_duplicates
    + records_removed_critical
)

cleaning_decisions = [
    "Exact duplicate rows were removed.",
    "Missing customer last names were retained without fabrication.",
    "Missing customer ZIP codes were retained without fabrication.",
    "No dedicated Order ID exists; customer IDs were not treated as unique order identifiers.",
    "Numeric columns were converted using safe numeric conversion.",
    "Infinite numeric values were converted to missing values and documented.",
    "Negative shipping durations were treated as critical invalid delivery records.",
    "Zero scheduled shipping durations were investigated and retained.",
    "Extreme shipping durations were flagged and retained.",
    "Categorical values were standardized without translating country or state names.",
    "Invalid geographic coordinates were flagged and retained for review.",
    "Late delivery risk was validated against values 0 and 1.",
    "Negative profit values were retained because they may represent legitimate losses.",
    "Financial outliers were flagged and retained.",
    "Personal customer identifiers were excluded from the analytical dataset and row-level reports.",
    "Only critical invalid delivery records were excluded."
]

cleaning_summary = {
    "project_name": "Analysis in Global Supply Chain Operations",
    "source_file": str(RAW_FILE),
    "cleaned_file": str(CLEANED_FILE),
    "execution_timestamp": datetime.now().isoformat(),
    "original_records": int(original_count),
    "original_columns": int(original_columns),
    "records_removed_as_exact_duplicates": records_removed_duplicates,
    "records_excluded_for_critical_delivery_errors": records_removed_critical,
    "total_records_removed": records_removed_total,
    "final_records": int(len(df_clean)),
    "final_columns": int(len(df_clean.columns)),
    "record_count_reconciled": bool(record_count_reconciled),
    "missing_customer_last_names": int(lname_missing_count),
    "missing_customer_zipcodes": int(zipcode_missing_count),
    "identifier_mismatches": int(identifier_mismatch_count),
    "zero_scheduled_shipping_durations": int(zero_scheduled_mask.sum()),
    "extreme_actual_shipping_durations": int(actual_extreme_mask.sum()),
    "extreme_scheduled_shipping_durations": int(scheduled_extreme_mask.sum()),
    "records_with_validation_flags": int(records_with_flags_mask.sum()),
    "critical_invalid_records_excluded": int(critical_invalid_count),
    "financial_outliers": {
        column: int(mask.sum())
        for column, mask in financial_outlier_masks.items()
    },
    "personal_columns_removed": removed_personal_columns,
    "cleaning_decisions": cleaning_decisions,
    "validation_results": {
        row["Validation"]: {
            "status": row["Status"],
            "details": row["Details"]
        }
        for row in quality_checks
    },
    "output_files": {
        "cleaned_dataset": str(CLEANED_FILE),
        "missing_value_report": str(MISSING_REPORT_FILE),
        "duplicate_report": str(DUPLICATE_REPORT_FILE),
        "invalid_records_report": str(INVALID_REPORT_FILE),
        "data_quality_report": str(QUALITY_REPORT_FILE),
        "financial_outlier_report": str(OUTLIER_REPORT_FILE),
        "identifier_validation_report": str(IDENTIFIER_REPORT_FILE),
        "numeric_validation_report": str(NUMERIC_REPORT_FILE),
        "categorical_validation_report": str(CATEGORY_REPORT_FILE),
        "geographic_validation_report": str(GEOGRAPHIC_REPORT_FILE),
        "shipping_duration_report": str(DURATION_REPORT_FILE),
        "missing_customer_investigation_report": str(INVESTIGATION_REPORT_FILE)
    }
}

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        cleaning_summary,
        file,
        indent=4,
        ensure_ascii=False,
        allow_nan=False,
        default=str
    )

print(f"Original records: {original_count:,}")
print(f"Duplicate records removed: {records_removed_duplicates:,}")
print(f"Critical invalid records excluded: {records_removed_critical:,}")
print(f"Total records removed: {records_removed_total:,}")
print(f"Final records: {len(df_clean):,}")
print(f"Final columns: {len(df_clean.columns)}")
print(f"Cleaning summary saved: {SUMMARY_FILE}")

# Final output verification

print("\nOUTPUT FILE VERIFICATION")
print("-" * 90)

output_files = [
    CLEANED_FILE,
    MISSING_REPORT_FILE,
    DUPLICATE_REPORT_FILE,
    INVALID_REPORT_FILE,
    QUALITY_REPORT_FILE,
    SUMMARY_FILE,
    OUTLIER_REPORT_FILE,
    IDENTIFIER_REPORT_FILE,
    NUMERIC_REPORT_FILE,
    CATEGORY_REPORT_FILE,
    GEOGRAPHIC_REPORT_FILE,
    DURATION_REPORT_FILE,
    INVESTIGATION_REPORT_FILE
]

output_verification = []

for file_path in output_files:
    output_verification.append({
        "File": str(file_path),
        "Exists": file_path.exists(),
        "Size (Bytes)": (
            file_path.stat().st_size
            if file_path.exists()
            else 0
        )
    })

output_verification = pd.DataFrame(
    output_verification
)

display(output_verification)

all_files_saved = bool(
    output_verification["Exists"].all()
)

all_quality_checks_passed = bool(
    (quality_report["Status"] == "PASS").all()
)

print("\nFINAL CLEANING STATUS")
print("=" * 90)

print(
    f"Data quality checks passed: "
    f"{int((quality_report['Status'] == 'PASS').sum())}"
)

print(
    f"Data quality checks failed: "
    f"{int((quality_report['Status'] == 'FAIL').sum())}"
)

print(
    f"All output files saved: "
    f"{'PASS' if all_files_saved else 'FAIL'}"
)

print(
    f"Record count reconciliation: "
    f"{'PASS' if record_count_reconciled else 'FAIL'}"
)

if (
    all_files_saved
    and all_quality_checks_passed
    and record_count_reconciled
):
    print("\nDATA CLEANING AND VALIDATION COMPLETED SUCCESSFULLY")
else:
    print("\nDATA CLEANING COMPLETED WITH VALIDATION ISSUES")
    print("Review the failed checks and generated reports before proceeding.")

print("=" * 90)


DATA CLEANING AND VALIDATION

SOURCE DATASET INFORMATION
------------------------------------------------------------------------------------------
Dataset path: C:\Users\hp\Analysis in Global Supply Chain Operations\data\raw\APL_Logistics.xlsx
Original records: 180,519
Original columns: 40
Required columns found: 40
Missing required columns: []

MISSING VALUE ANALYSIS
------------------------------------------------------------------------------------------


,Column,Missing Count,Missing Percentage,Non-Missing Count,Data Type
0,Customer Lname,8,0.0044,180511,object
1,Customer Zipcode,3,0.0017,180516,float64
2,Type,0,0.0000,180519,object
3,Order Item Profit Ratio,0,0.0000,180519,float64
4,Order City,0,0.0000,180519,object
5,Order Country,0,0.0000,180519,object
6,Order Customer Id,0,0.0000,180519,int64
7,Order Item Discount,0,0.0000,180519,float64
8,Order Item Discount Rate,0,0.0000,180519,float64
9,Order Item Product Price,0,0.0000,180519,float64


Missing-value report saved: C:\Users\hp\Analysis in Global Supply Chain Operations\outputs\reports\missing_value_report.csv

MISSING CUSTOMER INFORMATION INVESTIGATION
------------------------------------------------------------------------------------------
Missing Customer Lname records: 8
Missing Customer Zipcode records: 3
Customer personal information has not been reproduced in the investigation report.
Privacy-safe investigation report saved: C:\Users\hp\Analysis in Global Supply Chain Operations\outputs\reports\missing_customer_investigation_report.csv

MISSING VALUE HANDLING
------------------------------------------------------------------------------------------
Missing customer names and ZIP codes are retained in the source audit.
No artificial customer information has been generated.
Missing analytical fields are evaluated separately.
No records are removed solely because customer descriptive information is missing.

DUPLICATE RECORD VALIDATION
-----------------------------

,Identifier,Missing Values,Unique Values,Repeated Records,Unique Identifier Check
0,Customer Id,0,20652.0,159867.0,REPEATED CUSTOMER IDENTIFIERS
1,Order Customer Id,0,20652.0,159867.0,REPEATED CUSTOMER IDENTIFIERS
2,Customer Id vs Order Customer Id,0,NaN,NaN,0 mismatches


Important: The source dataset does not contain a dedicated Order ID column. Customer identifiers are not treated as unique order identifiers.

NUMERIC DATA TYPE VALIDATION
------------------------------------------------------------------------------------------


,Column,Original Data Type,Final Data Type,Conversion Failures,Missing Values After Conversion
0,Days for shipping (real),int64,int64,0,0
1,Days for shipment (scheduled),int64,int64,0,0
2,Benefit per order,float64,float64,0,0
3,Sales per customer,float64,float64,0,0
4,Late_delivery_risk,int64,int64,0,0
5,Category Id,int64,int64,0,0
6,Customer Id,int64,int64,0,0
7,Customer Zipcode,float64,float64,0,3
8,Department Id,int64,int64,0,0
9,Latitude,float64,float64,0,0


... (13 outputs left out)

,Validation,Status,Details
0,Required source columns available,PASS,Missing columns: []
1,No exact duplicate records in cleaned data,PASS,Duplicate count: 0
2,Actual shipping duration is numeric,PASS,int64
3,Scheduled shipping duration is numeric,PASS,int64
4,No negative actual shipping durations,PASS,Minimum: 0
5,No negative scheduled shipping durations,PASS,Minimum: 0
6,Late delivery risk contains only valid values,PASS,"Values: [0, 1]"
7,Delivery status values are standardized,PASS,"Values: ['Advance shipping', 'Late delivery', ..."
8,Shipping mode values are standardized,PASS,"Values: ['First Class', 'Same Day', 'Second Cl..."
9,Customer segment values are standardized,PASS,"Values: ['Consumer', 'Corporate', 'Home Office']"



DATA CLEANING SUMMARY
------------------------------------------------------------------------------------------
Original records: 180,519
Duplicate records removed: 0
Critical invalid records excluded: 0
Total records removed: 0
Final records: 180,519
Final columns: 32
Cleaning summary saved: C:\Users\hp\Analysis in Global Supply Chain Operations\outputs\reports\cleaning_summary.json

OUTPUT FILE VERIFICATION
------------------------------------------------------------------------------------------


,File,Exists,Size (Bytes)
0,C:\Users\hp\Analysis in Global Supply Chain Op...,True,49116556
1,C:\Users\hp\Analysis in Global Supply Chain Op...,True,1554
2,C:\Users\hp\Analysis in Global Supply Chain Op...,True,34
3,C:\Users\hp\Analysis in Global Supply Chain Op...,True,4449912
4,C:\Users\hp\Analysis in Global Supply Chain Op...,True,1532
5,C:\Users\hp\Analysis in Global Supply Chain Op...,True,7327
6,C:\Users\hp\Analysis in Global Supply Chain Op...,True,331
7,C:\Users\hp\Analysis in Global Supply Chain Op...,True,263
8,C:\Users\hp\Analysis in Global Supply Chain Op...,True,881
9,C:\Users\hp\Analysis in Global Supply Chain Op...,True,287



FINAL CLEANING STATUS
Data quality checks passed: 16
Data quality checks failed: 0
All output files saved: PASS
Record count reconciliation: PASS

DATA CLEANING AND VALIDATION COMPLETED SUCCESSFULLY
